In [39]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)


In [40]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2026-03-01"
sim_start = "2026-03-01"
end = None

In [41]:
RSI_DIV_CFG = {
    "length": 14,
    "pivot_lookback": 5,
    "min_rsi_delta": 2,

    # divergence filtering
    "os_level": 30,
    "ob_level": 70,
    "zone_mode": "cross",

    # panel styling
    "major_levels": [30, 70],
    "minor_levels": [20, 80],
    "show_zone_shading": True,

    # divergence labels
    "show_div_labels": True,
    "max_div_labels": 6,
    "label_font_size": 10,
    "label_xshift": 10,
    "label_yshift": 14,

    # main chart markers
    "mark_price": True,
    "price_marker_size": 18,
    "marker_offset_mult": 1.35,
}

In [42]:
ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]


ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]


ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),
]

In [43]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

merged = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
    },
    base_label="1m",
)

[20:45:53] INFO | 01. Parameters: market=spot, symbol=BTCUSDT, interval=1m, start=2026-03-01 00:00:00+00:00, end=LATEST
[20:45:53] INFO | 02. Cache directory: C:\Users\Talal Zahid\Desktop\Talal\Binance\data (cwd=C:\Users\Talal Zahid\Desktop\Talal\Binance)
[20:45:53] INFO | 03. Main cache file: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\binance_spot_BTCUSDT_1m.parquet
[20:45:53] INFO | 04. Partial checkpoint dir: C:\Users\Talal Zahid\Desktop\Talal\Binance\data\.partials\binance_spot_BTCUSDT_1m
[20:45:53] INFO | 05. Cache: HIT | rows=195,346 (main_rows=195,346, partial_files=0) | range=2026-01-01 00:00:00+00:00 -> 2026-05-16 15:45:00+00:00
[20:45:53] INFO | 06. No API fetch needed; local cache already covers requested range.
[20:45:54] INFO | 07. Cache consolidated and saved (parquet) | rows=195,346 | range=2026-01-01 00:00:00+00:00 -> 2026-05-16 15:45:00+00:00 | cleared_partial_files=0
[20:45:54] INFO | 08. Fetch summary | new_rows=0 | successful_pages=0 | requests=0 | retries=0 | 

In [44]:
for col in merged.columns:
    print(col)

open_time
open
high
low
close
volume
quote_volume
num_trades
taker_buy_base
taker_buy_quote
t
1m__ema__EMA_50
1m__ema__EMA_100
1m__ema__EMA_150
1m__ema__EMA_200
1m__ema__EMA_250
1m__macd_8_21_5__MACD
1m__macd_8_21_5__SIGNAL
1m__macd_8_21_5__HIST
1m__rsi14__RSI
1m__rsi14__BULL_DIV
1m__rsi14__BEAR_DIV
1m__rsi14__BULL_START_RSI
1m__rsi14__BEAR_START_RSI
1m__rsi14__BULL_RSI_LINE
1m__rsi14__BEAR_RSI_LINE
5m__ema__EMA_50
5m__ema__EMA_75
5m__ema__EMA_100
5m__ema__EMA_125
5m__ema__EMA_150
5m__ema__EMA_175
5m__ema__EMA_200
5m__macd_8_21_5__MACD
5m__macd_8_21_5__SIGNAL
5m__macd_8_21_5__HIST
5m__rsi14__RSI
5m__rsi14__BULL_DIV
5m__rsi14__BEAR_DIV
5m__rsi14__BULL_START_RSI
5m__rsi14__BEAR_START_RSI
5m__rsi14__BULL_RSI_LINE
5m__rsi14__BEAR_RSI_LINE
15m__ema__EMA_50
15m__ema__EMA_75
15m__ema__EMA_100
15m__ema__EMA_125
15m__ema__EMA_150
15m__ema__EMA_175
15m__ema__EMA_200
15m__macd_8_21_5__MACD
15m__macd_8_21_5__SIGNAL
15m__macd_8_21_5__HIST
15m__rsi14__RSI
15m__rsi14__BULL_DIV
15m__rsi14__BEAR_DIV
15

In [45]:
### 1 Minute Indicators ###
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"


### 5 Minute Indicators ###
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"


### 15 Minute Indicators ###
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"


LONG_EMA_FILTERS = [
    EMA50_1m,
    EMA100_1m,
    EMA150_1m,
    EMA100_5m,
    EMA100_15m,
]

In [46]:
N_CONFIRM = 1            # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 10
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 50 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

In [47]:
ema_price_confirmation_long = ALL(
    Rule(
        f"Last {N_CONFIRM} closes above all EMA filters",
        lambda c: c.last_n_closes_above_all(
            LONG_EMA_FILTERS,
            n=N_CONFIRM,
        )
    ),

    Rule(
        f"Last {N_CONFIRM} candles green",
        lambda c: c.consecutive_green(n=N_CONFIRM)
    ),
)

In [48]:
ema_ribbon_1m_bullish = Rule(
    "1m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
    lambda c: c.refs_ordered(
        [EMA50_1m, EMA100_1m, EMA150_1m],
        direction="desc",
        strict=True,
    )
)

ema_ribbon_5m_bullish = Rule(
    "5m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
    lambda c: c.refs_ordered(
        [EMA50_5m, EMA100_5m, EMA150_5m],
        direction="desc",
        strict=True,
    )
)

ema_ribbon_15m_bullish = Rule(
    "15m EMA ribbon bullish: EMA50 > EMA100 > EMA150",
    lambda c: c.refs_ordered(
        [EMA50_15m, EMA100_15m, EMA150_15m],
        direction="desc",
        strict=True,
    )
)


ema_ribbon_confirmation_long = ALL(
    ema_ribbon_1m_bullish,
    # ANY(
        # ema_ribbon_5m_bullish,
    ema_ribbon_15m_bullish,
    # )
)

In [49]:
macd_1m_above_threshold = Rule(
    "1m MACD above threshold",
    lambda c: c.gt(MACD_1M, MACD_THRESHOLD_1M)
)

macd_5m_above_threshold = Rule(
    "5m MACD above threshold",
    lambda c: c.gt(MACD_5M, MACD_THRESHOLD_5M)
)

macd_15m_above_threshold = Rule(
    "15m MACD above threshold",
    lambda c: c.gt(MACD_15M, MACD_THRESHOLD_15M)
)


macd_direction_confirmation_long = ALL(
    macd_1m_above_threshold,
    macd_5m_above_threshold,
    macd_15m_above_threshold,
)

In [50]:
macd_hist_1m_strong = Rule(
    "1m MACD - Signal above minimum",
    lambda c: c.gt(MACD_HIST_1M, MIN_MACD_DIFF_1M)
)

macd_hist_5m_strong = Rule(
    "5m MACD - Signal above minimum",
    lambda c: c.gt(MACD_HIST_5M, MIN_MACD_DIFF_5M)
)

macd_hist_15m_strong = Rule(
    "15m MACD - Signal above minimum",
    lambda c: c.gt(MACD_HIST_15M, MIN_MACD_DIFF_15M)
)


macd_strength_confirmation_long = ALL(
    macd_hist_1m_strong,
    macd_hist_5m_strong,
    macd_hist_15m_strong,
)

In [51]:
open_rules_long = ALL(
    ema_price_confirmation_long,
    ema_ribbon_confirmation_long,
    macd_direction_confirmation_long,
    macd_strength_confirmation_long,
)

In [52]:
close_rules_long = ALL(
    Rule(
        "Close below 15m EMA100",
        lambda c: c.lt("close", EMA100_15m)
    ),
)

In [53]:
from research.ema_move_analyzer import EmaMoveSpec, analyze_ema_move_strength

study_specs = [
    EmaMoveSpec(
        name="ema50_1m_vs_ema100_1m",
        fast_col="1m__ema__EMA_50",
        slow_col="1m__ema__EMA_100",
        dominance_cols=[
            "5m__ema__EMA_100",
            "15m__ema__EMA_100",
        ],
    ),
]

ema_report = analyze_ema_move_strength(
    df=merged,
    specs=study_specs,
    slope_bars=(3, 5, 10, 15, 30),
    future_bars=(5, 10, 15, 30, 60),
    target_move_pct=0.20,
    min_events=50,
    verbose=True,
)


[EMA ANALYZER | 0.0s] Started | rows=110,386 | specs=1 | slope_bars=(3, 5, 10, 15, 30) | future_bars=(5, 10, 15, 30, 60) | jobs=25
[EMA ANALYZER | 0.0s] Adding EMA move features and forward-return labels...
[EMA ANALYZER | 0.1s] Feature generation complete | cols=98
[EMA ANALYZER | 0.1s] Processing spec 1/1 | ema50_1m_vs_ema100_1m
[EMA ANALYZER | 0.1s] Job 1/25 | slope=ema50_1m_vs_ema100_1m__slope_pct_3 | spread=ema50_1m_vs_ema100_1m__spread_pct | fwd_bars=5
[EMA ANALYZER | 0.2s]   bucket stats done | rows=10
[EMA ANALYZER | 0.3s]   static grid done | rows=24
[EMA ANALYZER | 3.4s]   dynamic grid done | rows=36 | job_time=3.3s
[EMA ANALYZER | 3.4s] Job 2/25 | slope=ema50_1m_vs_ema100_1m__slope_pct_3 | spread=ema50_1m_vs_ema100_1m__spread_pct | fwd_bars=10
[EMA ANALYZER | 3.4s]   bucket stats done | rows=10
[EMA ANALYZER | 3.5s]   static grid done | rows=24
[EMA ANALYZER | 6.7s]   dynamic grid done | rows=36 | job_time=3.3s
[EMA ANALYZER | 6.7s] Job 3/25 | slope=ema50_1m_vs_ema100_1m__sl

In [59]:

ema_features = ema_report["features_df"]
static_thresholds = ema_report["static_thresholds"]
dynamic_thresholds = ema_report["dynamic_thresholds"]
bucket_stats = ema_report["bucket_stats"]

In [65]:
# bucket_stats
bucket_stats.sort_values(
    ["win_rate"],
    ascending=False,
)

,feature_col,fwd_bars,bucket,events,feature_min,feature_median,feature_max,avg_fwd_close_ret,median_fwd_close_ret,win_rate,avg_max_up,avg_max_down
180,ema50_1m_vs_ema100_1m__slope_pct_15,30,"(-0.864, -0.112]",11030,-0.863320,-0.164796,-0.112111,0.042965,0.050637,57.316410,0.296104,-0.290091
130,ema50_1m_vs_ema100_1m__slope_pct_10,30,"(-0.628, -0.0767]",11030,-0.627483,-0.112951,-0.076680,0.041420,0.049864,57.135086,0.296122,-0.293161
80,ema50_1m_vs_ema100_1m__slope_pct_5,30,"(-0.334, -0.0395]",11031,-0.332943,-0.058235,-0.039521,0.038818,0.049524,56.957665,0.295187,-0.296409
220,ema50_1m_vs_ema100_1m__slope_pct_30,15,"(-1.339, -0.21]",11030,-1.338185,-0.302936,-0.209607,0.024052,0.035213,56.935630,0.209788,-0.204506
30,ema50_1m_vs_ema100_1m__slope_pct_3,30,"(-0.214, -0.024]",11031,-0.213470,-0.035338,-0.024018,0.036126,0.049454,56.803554,0.295913,-0.299995
...,...,...,...,...,...,...,...,...,...,...,...,...
129,ema50_1m_vs_ema100_1m__slope_pct_10,15,"(0.0782, 1.174]",11032,0.078212,0.117708,1.174363,-0.004111,-0.027797,44.878535,0.242655,-0.219932
169,ema50_1m_vs_ema100_1m__slope_pct_15,10,"(0.114, 1.582]",11032,0.114401,0.171397,1.582407,-0.005125,-0.022608,44.869471,0.193579,-0.179620
189,ema50_1m_vs_ema100_1m__slope_pct_15,30,"(0.114, 1.582]",11030,0.114403,0.171405,1.582407,-0.005146,-0.037435,44.868540,0.341862,-0.312752
79,ema50_1m_vs_ema100_1m__slope_pct_5,15,"(0.0403, 0.627]",11032,0.040296,0.060441,0.626997,-0.001114,-0.028548,44.851342,0.247414,-0.218424


In [62]:
static = ema_report["static_thresholds"].copy()
dynamic = ema_report["dynamic_thresholds"].copy()

static_good = static[
    (static["events"] >= 100)
    & (static["median_fwd_close_ret"] > 0)
    & (static["avg_fwd_close_ret"] > 0)
].copy()

dynamic_good = dynamic[
    (dynamic["events"] >= 100)
    & (dynamic["median_fwd_close_ret"] > 0)
    & (dynamic["avg_fwd_close_ret"] > 0)
].copy()

static_best = static_good.sort_values(
    ["target_hit_rate", "median_fwd_close_ret", "avg_max_up", "score"],
    ascending=False,
)

dynamic_best = dynamic_good.sort_values(
    ["target_hit_rate", "median_fwd_close_ret", "avg_max_up", "score"],
    ascending=False,
)

static_best.head(20)

# static_best[
#     [
#         "slope_col",
#         "spread_col",
#         "fwd_bars",
#         "slope_threshold",
#         "spread_threshold",
#         "events",
#         "avg_fwd_close_ret",
#         "median_fwd_close_ret",
#         "win_rate",
#         "target_hit_rate",
#         "avg_max_up",
#         "avg_max_down",
#         "score",
#     ]
# ].head(20)

,slope_col,spread_col,fwd_bars,slope_threshold,spread_threshold,events,avg_fwd_close_ret,median_fwd_close_ret,win_rate,target_hit_rate,avg_max_up,avg_max_down,avg_winner,avg_loser,score


In [64]:
def get_best_ema_settings(
    report,
    mode="dynamic",
    min_events=100,
    min_win_rate=50,
    min_median_return=0,
    top_n=20,
):
    table = report["dynamic_thresholds"] if mode == "dynamic" else report["static_thresholds"]

    if table.empty:
        return table

    good = table.copy()

    good = good[
        (good["events"] >= min_events)
        & (good["win_rate"] >= min_win_rate)
        & (good["median_fwd_close_ret"] > min_median_return)
        & (good["avg_fwd_close_ret"] > 0)
    ].copy()

    if good.empty:
        return good

    sort_cols = [
        "target_hit_rate",
        "median_fwd_close_ret",
        "avg_fwd_close_ret",
        "avg_max_up",
        "score",
    ]

    return good.sort_values(sort_cols, ascending=False).head(top_n)

best_static = get_best_ema_settings(
    ema_report,
    mode="static",
    min_events=100,
    min_win_rate=52,
    min_median_return=0,
    top_n=20,
)

best_static

# best_dynamic = get_best_ema_settings(
#     ema_report,
#     mode="dynamic",
#     min_events=100,
#     min_win_rate=52,
#     min_median_return=0,
#     top_n=20,
# )

# best_dynamic

,slope_col,spread_col,fwd_bars,slope_threshold,spread_threshold,events,avg_fwd_close_ret,median_fwd_close_ret,win_rate,target_hit_rate,avg_max_up,avg_max_down,avg_winner,avg_loser,score


In [57]:
strategy = Strategy(
    open_rules_long=open_rules_long,
    close_rules_long=close_rules_long,

    # allow_short=True,
    # open_rules_short=open_rules_short,
    # close_rules_short=close_rules_short,
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=SimConfig(
        initial_cash=10_000,
        max_open_trades=1,
        fee_bps=10,
        slippage_bps=1,
        # sim_start=sim_start,
        sim_start="2026-01-01",
        sim_end="2026-05-09",
        sim_tz=tz,
    ),
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

[20:47:56] INFO | Simulation window applied | rows=100,500/110,386 | range=2026-03-01 05:00:00+05:00 -> 2026-05-09 23:59:00+05:00


Simulation:   0%|          | 0/100500 [00:00<?, ?bar/s]

[20:47:56] INFO | Simulation started | bars=100,500 | initial_cash=10,000.00 | cash_per_trade=10,000.00 | max_open_trades=1 | fee_bps=10 | slippage_bps=1
[20:48:02] INFO | Force-closing 1 open trade(s) at final bar.
[20:48:02] INFO | Simulation finished | elapsed=6.47s | bars=100,500 | opened=37 | closed=37 | final_cash=9,450.98 | total_return=-5.49% | max_dd=12.20%


{'initial_cash': 10000.0,
 'final_cash': 9450.981558271567,
 'total_pnl': -549.0184417284323,
 'total_return_pct': -5.490184417284327,
 'num_trades': 37.0,
 'num_winners': 13.0,
 'num_losers': 24.0,
 'num_breakeven': 0.0,
 'win_rate_pct': 35.13513513513514,
 'loss_rate_pct': 64.86486486486487,
 'gross_profit': 1453.6694620647872,
 'gross_loss': 2002.6879037932197,
 'profit_factor': 0.7258592111688715,
 'avg_pnl': -14.838336262930603,
 'median_pnl': -55.33458803105658,
 'avg_winner': 111.82072785113748,
 'avg_loser': -83.44532932471749,
 'avg_loser_abs': 83.44532932471749,
 'largest_winner': 433.1469455273112,
 'largest_loser': -211.0916708638889,
 'payoff_ratio': 1.3400477744656087,
 'expectancy_per_trade': -14.838336262930603,
 'expectancy_pct_initial_cash': -0.14838336262930604,
 'avg_return_pct': -0.14838336262930601,
 'median_return_pct': -0.5533458803105659,
 'best_return_pct': 4.331469455273112,
 'worst_return_pct': -2.110916708638889,
 'avg_duration_min': 699.8108108108108,
 'me

In [ ]:
# fig1 = plot_balance_by_trade(res.trades, initial_cash=10_000, interval="5m")
# fig1.show()

# fig2 = plot_trade_pnl_bars(res.trades, initial_cash=10_000, interval="1m")
# fig2.show()


In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
# plot_indicators = make_plot_indicators(
#     base_specs=ind_1m,
#     mtf_specs={
#         "5m": ind_5m,
#         "15m": ind_15m,
#     },
#     mtf_include={
#         "5m": ["macd_8_21_5"],
#         "15m": ["macd_8_21_5"],
#     }
# )

In [ ]:
indicators_by_tf = {
    "1m": ind_1m,
    "5m": ind_5m,
    "15m": ind_15m,
}

plot_toggles = [
    PlotToggle("1m", "ema", visible=True),
    PlotToggle("5m", "ema", visible=True),
    PlotToggle("15m", "ema", visible=True),


    # Available in legend, hidden by default
    PlotToggle("1m", "macd_8_21_5", visible="legendonly"),
    PlotToggle("5m", "macd_8_21_5", visible="legendonly"),
    PlotToggle("15m", "macd_8_21_5", visible="legendonly"),

    # Not plotted at all, but still can be used in rules
    PlotToggle("1m", "rsi14", visible=False),
    PlotToggle("5m", "rsi14", visible=False),
    PlotToggle("15m", "rsi14", visible=False),
]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)

assert_plot_columns_exist(merged, plot_indicators)

In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        #date_from="2026-05-06",
        #date_to="2026-05-10",
        date_from="2026-05-01",
        date_to="2026-05-06",
        height=1400,
        width=1900,
    ),
)

# plot_simulation(
#     df_raw=df1_feat,  # (ideally raw candles df; but if yours works as-is, ok)
#     symbol=symbol,
#     interval="5m",
#     market=market,
#     trades=res.trades,
#     ma_windows=[20],
#     indicators=ind_5m,
#     plot_cfg=PlotConfig(
#         tz="Asia/Karachi",
#         date_from="2026-02-01",
#         date_to="2026-02-05",
#         height=1400,
#         width=1900
#     ),
# )